In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_path = '/content/drive/My Drive/PlantVillageDataset/PlantVillage'

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout, Activation


In [ ]:
 #run only once ----> train + test split

# import os
# import shutil
# import random


# dataset_path = '/content/drive/MyDrive/PlantVillageDataset/PlantVillage'

# test_path = '/content/drive/MyDrive/PlantVillageDataset/PlantVillage_test'


# if not os.path.exists(test_path):
#     os.makedirs(test_path)

# test_split = 0.20

# for class_name in os.listdir(dataset_path):
#     class_folder = os.path.join(dataset_path, class_name)
#     if not os.path.isdir(class_folder):
#         continue

#     test_class_folder = os.path.join(test_path, class_name)
#     if not os.path.exists(test_class_folder):
#         os.makedirs(test_class_folder)

#     images = os.listdir(class_folder)
#     random.shuffle(images)

#     num_test = int(len(images) * test_split)
#     test_images = images[:num_test]

#     for img in test_images:
#         src = os.path.join(class_folder, img)
#         dst = os.path.join(test_class_folder, img)
#         shutil.move(src, dst)

# print("Dataset successfully split into training and test set.")


In [ ]:
train_data_generator = ImageDataGenerator(
    rescale = 1/.255,
    rotation_range = 20,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.1,
    zoom_range = 0.1,
    horizontal_flip = True,
    fill_mode = 'nearest',
    validation_split = 0.2
)

train_generator = train_data_generator.flow_from_directory(
    dataset_path,
    target_size = (227,227),
    batch_size = 32,
    class_mode = 'categorical',
    subset = 'training'
)

val_generator = train_data_generator.flow_from_directory(
    dataset_path,
    target_size = (227,227),
    batch_size = 32,
    class_mode = 'categorical',
    subset = 'validation'
)

Found 11240 images belonging to 15 classes.
Found 2804 images belonging to 15 classes.


In [ ]:
#AlexNet architecture defined

model = Sequential()

#1st convolution layer
model.add(Conv2D(96,kernel_size=(11,11),strides=(4,4),input_shape=(227,227,3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(3,3),strides=(2,2)))

#2nd convolution layer
model.add(Conv2D(256,kernel_size=(5,5),padding='same'))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(3,3),strides=(2,2)))

#3rd convolution layer
model.add(Conv2D(384,kernel_size=(3,3),padding='same'))
model.add(Activation('relu'))

#4th convolution layer
model.add(Conv2D(384,kernel_size=(3,3),padding='same'))
model.add(Activation('relu'))

#5th convolution layer
model.add(Conv2D(256,kernel_size=(3,3),padding='same'))
model.add(MaxPool2D(pool_size=(3,3),strides=(2,2)))

#Flatten layer
model.add(Flatten())

#Dense layer 1
model.add(Dense(4096))
model.add(Activation('relu'))
model.add(Dropout(0.5))

#dense layer 2
model.add(Dense(4096))
model.add(Activation('relu'))
model.add(Dropout(0.5))

# output layer

model.add(Dense(15))
model.add(Activation('softmax'))





In [ ]:
#compile the model
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
# save the model weights after each epoch
checkpoint_path = '/content/drive/My Drive/PlantVillageDataset/best_model.weights.h5'

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath = checkpoint_path,
    save_weights_only=True,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose = 1
)


In [ ]:
#load the latest exported model weights
model.load_weights('/content/drive/My Drive/PlantVillageDataset/epoch03.weights.h5')

In [ ]:
history = model.fit(train_generator,
                    validation_data=val_generator,
                    initial_epoch = 3,
                    epochs = 10,
                    callbacks=[checkpoint_cb])

Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.2135 - loss: 2.3624
Epoch 4: val_accuracy improved from -inf to 0.24572, saving model to /content/drive/My Drive/PlantVillageDataset/best_model.weights.h5
352/352 ━━━━━━━━━━━━━━━━━━━━ 2716s 8s/step - accuracy: 0.2135 - loss: 2.3624 - val_accuracy: 0.2457 - val_loss: 2.1859
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.2804 - loss: 2.1436
Epoch 5: val_accuracy improved from 0.24572 to 0.38766, saving model to /content/drive/My Drive/PlantVillageDataset/best_model.weights.h5
352/352 ━━━━━━━━━━━━━━━━━━━━ 2445s 7s/step - accuracy: 0.2805 - loss: 2.1434 - val_accuracy: 0.3877 - val_loss: 1.7985
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.3445 - loss: 1.9634
Epoch 6: val_accuracy did not improve from 0.38766
352/352 ━━━━━━━━━━━━━━━━━━━━ 2454s 7s/step - accuracy: 0.3444 - loss: 1.9637 - val_accuracy: 0.3516 - val_loss: 1.9594
Epoch 7/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0